# Design a Chat Application Like WhatsApp

**Company:** MongoDB (GothamLoop question bank) · **Category:** System Design · **Tags:** Onsite Loop, Caching, Concurrency, Databases, Distributed Systems, Networking · **Difficulty/Frequency:** Uncommon (3/10)

> **How to use this notebook.** System design has no algorithm to benchmark, so the code cells here hold a **runnable capacity model** instead. Every figure the source answer states is recomputed from its stated assumptions and pinned by an assertion — change an assumption and watch the conclusion move. That is the skill the interview is actually testing.

## Concepts

**What this question is really testing:**
- **Fan-out**: one write becoming k writes, and whether you pay that cost on write or on read
- Separating the **control plane** (who is connected where) from the **data plane** (durable storage)
- Whether your **numbers** justify your architecture, rather than decorating it

**First-principles primer — what is each piece?**

- **Fan-out on write vs. on read.** A message to a group of k people can be stored once (fan-out on read: every reader queries the conversation) or copied into k inboxes (fan-out on write: every reader queries only their own inbox). Chat is **read-heavy** — you open the app far more often than you send — so you pay once at write time to make every read O(1). The cost is storage amplification of exactly k.
- **The inbox table.** One row per `(user, message)`. It is what makes an offline user work at all: the message is already sitting in their inbox before they reconnect, so "delivery" is just a notification that something is there.
- **WebSocket vs. polling.** Polling costs you half the poll interval in latency, on average, and burns requests when nothing has changed. A long-lived connection makes push possible — which is the only way to hit a sub-500 ms target.
- **The gateway/chat split.** Gateways are *stateful* (they hold the socket) but hold no data. Chat services are *stateless* but own the data. Keeping connection state out of the database is what lets you scale the two independently.

**The insight the whole design rests on:**

> The server is a **dumb relay for ciphertext** that happens to know enough metadata to route it.

Everything follows from that. The server cannot search message content, cannot generate a message preview for a push notification, and cannot help a user recover history without their keys. Those are not oversights — they are the *price* of end-to-end encryption, and naming them is what shows you understand the trade rather than reciting it.

**Delivery is three separate events**, and conflating them is the most common mistake:

| Event | Means | Who observes it |
|---|---|---|
| **sent** | the server durably stored it | sender |
| **delivered** | the recipient's device has it | sender, via ACK |
| **read** | the recipient opened the conversation | sender, via read receipt |

The sender's ✓ appears at *sent*, not at *delivered*. If you acknowledge before the write is replicated, a crash loses a message the sender believes was sent.

**Simple worked example.** Alice sends "hi" to a group of 4:

```
1. Alice's client encrypts "hi"          -> ciphertext blob
2. Chat Service writes 1 row to messages -> durable. Alice sees ONE tick.
3. Chat Service writes 4 rows to inbox   -> fan-out on write
4. Publish MessageCreated to the queue
5. Delivery worker checks presence for each of the 4:
     Bob   online  -> push via his Gateway -> ACK -> inbox row = 'delivered'
     Carol offline -> do nothing; the row is already in her inbox
     Dan   offline -> send an APNs push ("New message") with NO content
     Alice online  -> her own copy, already 'read'
6. Bob opens the chat -> ReadReceipt -> inbox row = 'read' -> Alice sees two blue ticks
```

## Requirements & Scale

| Functional | Non-functional |
|---|---|
| 1:1 and group messaging | **< 500 ms** delivery latency |
| Presence (online/offline) | **99.99%** availability |
| Delivery + read receipts | Hundreds of millions of users |
| Media sharing | **No message loss** |
| End-to-end encryption | E2E encryption |
| Push notifications | |
| Multi-device history sync | |

**Stated scale:** 500M DAU · 50 messages/user/day.

Everything below is derived from exactly those two numbers plus assumptions we name explicitly.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "capacity.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

from capacity import (DAY, YEAR, KB, MB, GB, TB, PB,
                      human_bytes, human_count, human_rate,
                      table, assumption_table, sensitivity)

# ---- The two numbers the question gives us --------------------------------
DAU = 500_000_000
MESSAGES_PER_USER_PER_DAY = 50

# ---- Everything else is an ASSUMPTION we are choosing, and must defend -----
PEAK_MULTIPLIER = 4          # peak / average; the answer says "3-5x"
CONCURRENT_FRACTION = 0.12   # share of DAU online at once; the answer says "10-15%"
CONNS_PER_GATEWAY = 100_000  # WebSockets one gateway node can hold
AVG_FANOUT = 4               # inbox rows written per message (1:1 = 2, groups more)
INBOX_ROW_BYTES = 200
MESSAGE_ROW_BYTES = 200
MEDIA_SHARE = 0.10           # fraction of messages carrying media
AVG_MEDIA_BYTES = 500 * KB

assumption_table({
    "Daily active users":        human_count(DAU),
    "Messages per user per day": MESSAGES_PER_USER_PER_DAY,
    "Peak multiplier":           f"{PEAK_MULTIPLIER}x average",
    "Concurrent fraction":       f"{CONCURRENT_FRACTION:.0%} of DAU",
    "Connections per gateway":   human_count(CONNS_PER_GATEWAY),
    "Average fan-out":           f"{AVG_FANOUT} inbox rows per message",
    "Media share":               f"{MEDIA_SHARE:.0%} of messages",
    "Average media size":        human_bytes(AVG_MEDIA_BYTES),
})

## Capacity model — the numbers that decide the architecture

Three questions, and each one forces a specific design choice:

1. **How many messages per second?** → decides whether one queue can carry it.
2. **How many concurrent connections?** → decides how many gateway nodes you run.
3. **Text storage vs. media storage?** → decides that media never touches the database.

The third is the one that matters most, and the answer is not close.

In [ ]:
# ---- 1. Message rate -------------------------------------------------------
messages_per_day = DAU * MESSAGES_PER_USER_PER_DAY
msgs_per_sec_avg = messages_per_day / DAY
msgs_per_sec_peak = msgs_per_sec_avg * PEAK_MULTIPLIER

table([
    ("Messages per day",     human_count(messages_per_day)),
    ("Average rate",         human_rate(msgs_per_sec_avg)),
    (f"Peak ({PEAK_MULTIPLIER}x)", human_rate(msgs_per_sec_peak)),
], title="MESSAGE RATE")

# The source states ~289k/sec average and ~1M/sec peak. Check we reproduce that.
assert 280_000 < msgs_per_sec_avg < 295_000, msgs_per_sec_avg
assert 1_000_000 < msgs_per_sec_peak < 1_200_000, msgs_per_sec_peak

# ---- 2. Concurrent connections --------------------------------------------
concurrent = DAU * CONCURRENT_FRACTION
gateways = concurrent / CONNS_PER_GATEWAY

table([
    ("Peak concurrent connections", human_count(concurrent)),
    ("Gateway nodes needed",        f"{gateways:,.0f}"),
    ("...at 1M conns/node (tuned)", f"{concurrent / 1_000_000:,.0f}"),
], title="CONNECTIONS")

assert 50_000_000 <= concurrent <= 75_000_000, "the answer's stated 50-75M range"
assert 500 <= gateways <= 750, "the answer's stated 500-750 gateway nodes"

In [ ]:
# ---- 3. Storage: the ratio that decides everything ------------------------
inbox_rows_per_day = messages_per_day * AVG_FANOUT
inbox_bytes_per_day = inbox_rows_per_day * INBOX_ROW_BYTES
message_bytes_per_day = messages_per_day * MESSAGE_ROW_BYTES
text_bytes_per_day = inbox_bytes_per_day + message_bytes_per_day

media_msgs_per_day = messages_per_day * MEDIA_SHARE
media_bytes_per_day = media_msgs_per_day * AVG_MEDIA_BYTES

table([
    ("Inbox rows per day",   human_count(inbox_rows_per_day)),
    ("Inbox storage / day",  human_bytes(inbox_bytes_per_day)),
    ("Message rows / day",   human_bytes(message_bytes_per_day)),
    ("TEXT total / day",     human_bytes(text_bytes_per_day)),
    ("TEXT total / year",    human_bytes(text_bytes_per_day * 365)),
    ("", ""),
    ("Media messages / day", human_count(media_msgs_per_day)),
    ("MEDIA total / day",    human_bytes(media_bytes_per_day)),
    ("MEDIA total / year",   human_bytes(media_bytes_per_day * 365)),
    ("", ""),
    ("MEDIA / TEXT ratio",   f"{media_bytes_per_day / text_bytes_per_day:.0f}x"),
], title="STORAGE")

# The source's headline claims
assert abs(inbox_bytes_per_day - 20 * TB) / TB < 1, "inbox ~20 TB/day"
assert abs(text_bytes_per_day - 25 * TB) / TB < 1, "text ~25 TB/day"
assert abs(media_bytes_per_day - 1.25 * PB) / PB < 0.01, "media ~1.25 PB/day"
assert media_bytes_per_day / text_bytes_per_day > 40, "media dwarfs text by ~50x"

print(f"\n  => Media is {media_bytes_per_day / text_bytes_per_day:.0f}x text storage.")
print("     THIS is why media goes to object storage + CDN and never touches the DB.")

# ---- Bandwidth -------------------------------------------------------------
media_upload_bytes_per_sec = media_bytes_per_day / DAY
table([
    ("Media upload bandwidth", f"{media_upload_bytes_per_sec / GB:.1f} GB/s"),
], title="BANDWIDTH")
assert 13 < media_upload_bytes_per_sec / GB < 16, "the answer's stated ~14.5 GB/s"

### Which assumption is the estimate most sensitive to?

The point of a capacity estimate is not the number — it is knowing **which input the number hangs on**. Here, two assumptions were invented by us rather than given, and they move the answer by orders of magnitude:

- **Average fan-out** drives all text storage. It is the difference between "one database" and "a fleet".
- **Average media size** drives the total, full stop.

Anything else — row sizes, peak multiplier — is noise by comparison.

In [ ]:
sensitivity(lambda f: messages_per_day * f * INBOX_ROW_BYTES + message_bytes_per_day,
            AVG_FANOUT, "avg fan-out")

print()
sensitivity(lambda s: media_msgs_per_day * s,
            AVG_MEDIA_BYTES, "avg media size")

print()
sensitivity(lambda share: messages_per_day * share * AVG_MEDIA_BYTES,
            MEDIA_SHARE, "media share")

print("\n  => A group-heavy product (fan-out 20) is 5x the text storage.")
print("     A video-heavy product (2 MB average) is 4x the media storage.")
print("     Both dwarf any row-size tuning - so those are the numbers to pin down first.")

## Data model

The one design decision worth defending is the **`inbox` table**: one row per `(user, message)`.

That is deliberate storage amplification, and it buys three things at once:

- **Offline delivery** — a message is *already* in the recipient's inbox before they reconnect. "Delivery" is only a notification.
- **O(1) sync** — reconnecting is `SELECT ... WHERE user_id = ? AND message_id > ?`, hitting exactly one shard.
- **Per-user status** — `delivered`/`read` is a property of a *recipient*, not of a message, so it needs somewhere per-user to live.

Sharding follows the access pattern: `inbox` by `user_id` (a user's sync never crosses a shard), `messages` by `conversation_id` (loading a conversation's history never crosses a shard).

In [ ]:
SCHEMA = """
CREATE TABLE users (
    user_id BIGINT PRIMARY KEY,
    phone_number VARCHAR(20) UNIQUE,
    display_name VARCHAR(100),
    public_key TEXT,                       -- E2E identity key; the SERVER never has the private half
    created_at TIMESTAMP
);

CREATE TABLE conversations (
    conversation_id BIGINT PRIMARY KEY,
    type ENUM('one_on_one', 'group'),
    group_name VARCHAR(100),
    created_at TIMESTAMP
);

CREATE TABLE conversation_members (
    conversation_id BIGINT,
    user_id BIGINT,
    joined_at TIMESTAMP,
    last_read_message_id BIGINT DEFAULT 0,
    PRIMARY KEY (conversation_id, user_id)
);

CREATE TABLE messages (                    -- SHARDED BY conversation_id
    message_id BIGINT PRIMARY KEY,         -- monotonic per conversation => client sorts by it
    conversation_id BIGINT,
    sender_id BIGINT,
    message_type ENUM('text','image','video','file'),
    content TEXT,                          -- CIPHERTEXT. The server cannot read this.
    media_id BIGINT,                       -- media lives in object storage, not here
    created_at TIMESTAMP,
    INDEX idx_conversation_time (conversation_id, message_id)
);

CREATE TABLE inbox (                       -- SHARDED BY user_id. The fan-out target.
    user_id BIGINT,
    message_id BIGINT,
    conversation_id BIGINT,
    status ENUM('delivered','read'),       -- per-RECIPIENT, which is why this table exists
    delivered_at TIMESTAMP,
    read_at TIMESTAMP,
    PRIMARY KEY (user_id, message_id)      -- also the sync cursor: WHERE user_id=? AND message_id>?
);

CREATE TABLE media (
    media_id BIGINT PRIMARY KEY,
    uploader_id BIGINT,
    media_type ENUM('image','video','file'),
    object_key VARCHAR(255),               -- S3; the blob is encrypted CLIENT-side
    size_bytes BIGINT,
    encryption_key_id BIGINT,
    created_at TIMESTAMP
);
"""
print(SCHEMA)

# What one message actually costs to store, end to end.
one_to_one_rows = 2
group_of_10_rows = 10
table([
    ("1:1 message",        f"1 messages row + {one_to_one_rows} inbox rows"),
    ("Group of 10",        f"1 messages row + {group_of_10_rows} inbox rows"),
    ("Group of 100,000",   "1 messages row + 100,000 inbox rows  <- fan-out breaks down here"),
], title="WRITE AMPLIFICATION PER MESSAGE")

## The delivery pipeline, and where it can fail

```
  Alice ──WebSocket──> Gateway ──> Chat Service
                                        │
                                        ├─ 1. write `messages`   (durable, replicated)
                                        ├─ 2. write k `inbox` rows
                                        └─ 3. publish MessageCreated ──> Queue
                                                                          │
                                              Delivery worker <───────────┘
                                                     │
                                        ┌────────────┴────────────┐
                                   online?                    offline?
                                        │                         │
                              push via Gateway            already in inbox;
                                        │                 send APNs/FCM hint
                                      ACK
                                        │
                            inbox.status = 'delivered'
                                        │
                              notify Alice (two ticks)
```

**The ordering that matters:** steps 1–2 are **synchronous and replicated** before Alice's ACK. Step 3 is asynchronous. That split is what lets the sender's ✓ mean *"durably stored"* while the fan-out to a 1,000-member group happens in the background.

**Failure modes, and what each one costs:**

| Failure | Consequence | Mitigation |
|---|---|---|
| Crash before the inbox write | **message lost** | write durably *before* ACKing the sender |
| ACK lost after a push | duplicate delivery | client de-duplicates on `message_id` |
| Two gateways push out of order | jumbled conversation | client sorts by `message_id` |
| Gateway node dies | clients disconnect | reconnect elsewhere; sync from last `message_id` |
| Delivery workers fall behind | latency grows | monitor queue depth, autoscale consumers |

Note what is *not* in that table: exactly-once delivery. It is not offered. The system is **at-least-once**, and the client's `message_id` de-duplication is what makes that acceptable.

In [ ]:
def sender_tick_semantics():
    """What each tick actually means - the distinction interviewers probe."""
    table([
        ("One tick (sent)",      "the server durably stored it. NOT that anyone received it."),
        ("Two ticks (delivered)","the recipient's DEVICE has it and ACKed."),
        ("Blue ticks (read)",    "the recipient OPENED the conversation."),
    ], title="WHAT THE TICKS MEAN")


sender_tick_semantics()

# Latency budget: does the design actually fit under 500 ms?
budget = [
    ("Client encrypt",            5),
    ("Client -> Gateway (RTT/2)", 40),
    ("Gateway -> Chat Service",   2),
    ("Write messages (replicated)", 15),
    ("Write inbox rows",          10),
    ("Publish to queue",          5),
    ("Queue -> delivery worker",  20),
    ("Presence lookup (Redis)",   2),
    ("Worker -> recipient Gateway", 5),
    ("Gateway -> recipient (RTT/2)", 40),
    ("Client decrypt + render",   10),
]
total_ms = sum(ms for _, ms in budget)
table([(step, f"{ms:>4} ms") for step, ms in budget] + [("TOTAL", f"{total_ms:>4} ms")],
      title="LATENCY BUDGET (p50, same region)")

assert total_ms < 500, f"design must fit the stated <500ms target, got {total_ms}ms"
print(f"\n  => {total_ms} ms of a 500 ms budget. Headroom: {500 - total_ms} ms")
print("     Note the two network hops (80 ms) dominate. Cross-region would blow the budget,")
print("     which is the argument for regional gateway placement.")

## Discussion — the follow-ups, and what each is really asking

- **A group with 100,000 members.** Fan-out on write breaks: one message becomes 100,000 inbox rows, and a chatty group of that size generates more inbox writes than the rest of the system combined. The fix is to **switch strategies by group size** — small groups fan out on write, large groups fan out on read (store once; members query the conversation on open). This is precisely what Twitter does for celebrity accounts, and the hybrid is the answer: most groups are small, so you keep the O(1) read path where it is cheap and pay O(k) reads only where fan-out would be ruinous.
- **Ordering across multiple devices.** `message_id` is monotonic **per conversation**, assigned server-side, which is what gives every device the same total order regardless of network path. Client-side wall-clock timestamps would not — two devices' clocks disagree, and a message sent later can arrive first. The server-assigned sequence is the single source of ordering truth.
- **Search, when the server cannot decrypt.** This is the honest cost of E2E. The server can index **metadata only** — sender, conversation, timestamp — never content. Full-text search must happen **on-device**, over the locally decrypted history, which means it works only for messages that device has synced. A new device has nothing to search until it syncs, and it cannot sync without the keys. Acknowledging this limitation is worth more than inventing a scheme that quietly breaks the encryption guarantee.
- **A lost phone.** The keys were on the device, so the history is cryptographically unreachable. Two options, and both are trade-offs the user should make explicitly: an **encrypted cloud backup** whose key is derived from a user passphrase (recoverable, but now the passphrase is the weak point), or **no backup** (unrecoverable, but nothing to compel or leak). WhatsApp offers the first as opt-in for exactly this reason.
- **Spam and abuse without reading content.** You are left with **metadata and behaviour**: message rate, fan-out breadth, account age, how many recipients report a sender, how many of a sender's messages go to people who never reply. That is genuinely weaker than content filtering, and it is why E2E platforms lean on **user reporting** — a reported message is voluntarily decrypted *by the recipient* and forwarded, which preserves the guarantee while still producing evidence.

## Patterns learned

- **Fan-out is a choice about *when* you pay, not *whether*.** On write costs storage amplification of k; on read costs a k-way gather per open. Pick by read/write ratio — and for very large k, pick differently.
- **The sender's ACK must mean "durable", not "accepted".** Everything after that ACK can be asynchronous. Getting this boundary right is the difference between a system that loses messages and one that does not.
- **Delivered and read are separate events, both per-recipient.** That is *why* the inbox table exists — a per-message status field cannot express "Bob read it, Carol didn't".
- **Connection state and data belong in different tiers.** Gateways hold sockets and no data; chat services hold data and no sockets. Each scales on its own axis.
- **Presence is best-effort, and saying so is the answer.** Heartbeats are periodic, mobile OSes kill sockets, partitions happen. Claiming accurate presence signals you have not thought about it.
- **Do the storage arithmetic before choosing the storage.** Media at 50× text is not a detail — it is the entire justification for object storage plus CDN, and you cannot make that argument without the number.
- **End-to-end encryption is a constraint that propagates.** No server-side search, no message previews in push notifications, no history recovery without keys. Enumerating what it *forbids* demonstrates real understanding.
- **State your assumptions, then test their sensitivity.** Fan-out and media size swing this estimate by 5× and 4×. Row sizes swing it by nothing. Knowing which is which is the actual skill.